In [83]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


In [84]:
# Positional Encoding...

def get_angles(pos , i , d_model):
  angle_rates = 1/np.power(10000,((2*(i//2)) / np.float32(d_model)))
  return pos * angle_rates

def positional_encoding(position , d_model):
  angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                          np.arange(d_model)[np.newaxis ,:],
                          d_model)
  # apply sin to even index
  angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

  # apply cos to odd index
  angle_rads[:, 1::2] = np.sin(angle_rads[:, 1::2])

  # add batch dimension
  pos_encoding = angle_rads[np.newaxis, ...]

  return tf.cast(pos_encoding,dtype=tf.float32)

In [85]:
# Masking

def create_padding_mask(seq):
  ''' seq shape --> (batch_size , seq_length)
      output of this func dimn -->(batch_size , 1 , 1 , seq_length)
      output -->(batch_size , heads , Query_length , key_length)
       '''

  mask = tf.cast(tf.equal(seq,0),tf.float32)
  return mask[: , tf.newaxis , tf.newaxis , :]


def create_look_ahead_mask(size):
  ''' size --> target sequence length
      output of this func -->(size,size)
      '''
  i = tf.range(size)[:, tf.newaxis]
  j = tf.range(size)

  mask = tf.cast(i < j, tf.float32)

  return mask


  def create_mask(inp , tar):
    # encoder padding mask
    enc_padding_mask = create_padding_mask(inp)

    # decoder padding_mask(used in cross attention)
    dec_padding_mask = create_padding_mask(inp)

    # look ahead mask for the 1st attention block (masked MHA)
    look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
    dec_target_padding_mask = create_padding_mask(tar)
    combined_mask = tf.maximum(dec_target_padding_mask , look_ahead_mask)

    return enc_padding_mask , combined_mask , dec_padding_mask


In [86]:
# Scaled Dot Product Attention

def scaled_dot_product_attention(q,k,v,mask):

  """
    q : (batch_size, num_heads, seq_len_q, depth)
    k : (batch_size, num_heads, seq_len_k, depth)
    v : (batch_size, num_heads, seq_len_v, depth)
    mask : broadcastable to (batch_size, num_heads, seq_len_q, seq_len_k)

    Returns:
        output             : (..., seq_len_q, depth)
        attention_weights  : (..., seq_len_q, seq_len_k)
    """
  matmul_qk = tf.matmul(q,k,transpose_b = True)
  dk = tf.cast(tf.shape(k)[-1],tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

  if mask is not None:
    scaled_attention_logits += (mask * -1e9)

  attention_weights = tf.nn.softmax(scaled_attention_logits, axis = -1)
  output = tf.matmul(attention_weights,v)
  return output, attention_weights

In [87]:
class MultiHeadAttention(tf.keras.layers.Layer):
  def __init__(self,d_model,num_heads):
    super(MultiHeadAttention,self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model
    self.supports_masking = True # Declare support for masking

    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

    self.depth = d_model // num_heads
    self.wq = tf.keras.layers.Dense(d_model)
    self.wk = tf.keras.layers.Dense(d_model)
    self.wv = tf.keras.layers.Dense(d_model)

    # final projection
    self.dense = tf.keras.layers.Dense(d_model)

  def split_heads(self,x,batch_size):
    """
    Input:
        (batch_size, seq_len, d_model)

    Output:
        (batch_size, num_heads, seq_len, depth)

    Axis: 0    1       2      3
          B   Seq    Head  Depth
    """
    x = tf.reshape(x , (batch_size,-1,self.num_heads,self.depth))
    x = tf.transpose(x,perm = [0,2,1,3])
    return x

  def call(self,v,k,q,mask):
    batch_size = tf.shape(q)[0]

    q = self.wq(q)
    k = self.wk(k)
    v = self.wv(v)

    q = self.split_heads(q, batch_size)
    k = self.split_heads(k, batch_size)
    v = self.split_heads(v, batch_size)

    scaled_attention, attention_weights = (
            scaled_dot_product_attention(
                q,
                k,
                v,
                mask
            )
        )
    scaled_attention = tf.transpose(scaled_attention,perm = [0,2,1,3])

    concat_attention = tf.reshape(
            scaled_attention,
            (
                batch_size,
                -1,
                self.d_model
            )
        )
    output = self.dense(concat_attention)
    return output, attention_weights

In [88]:
def Feed_Forward_Network(d_model, dff):
    return tf.keras.Sequential(
        [
            tf.keras.layers.Dense(dff, activation="relu"),
            tf.keras.layers.Dense(d_model),
        ]
    )

In [89]:
class Encoder_Layer(tf.keras.layers.Layer):
  def __init__(self,d_model,num_heads,dff,dropout_rate=0.1):
    super().__init__()

    # Multi-Head Self Attention
    self.mha = MultiHeadAttention(d_model,num_heads)

    # Feed Forward Network
    self.ffn = Feed_Forward_Network(d_model,dff)

    # Layer Normalization
    self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    # Dropout
    self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
    self.dropout2 = tf.keras.layers.Dropout(dropout_rate)
    self.supports_masking = True # Declare support for masking

  def build(self, input_shape):
    super().build(input_shape)

  def call(self,x,training=False,mask=None):
    attn_output , attention_weights = self.mha(v=x,k=x,q=x,mask=mask)
    attn_output = self.dropout1(attn_output,training=training)
    out_1 = self.layernorm1(x + attn_output)

    ffn_output = self.ffn(out_1)
    ffn_output = self.dropout2(ffn_output,training=training)
    out_2 = self.layernorm2(out_1 + ffn_output)

    return out_2, attention_weights

In [90]:
class Encoder(tf.keras.layers.Layer):

    def __init__(self,
                 num_layers,
                 d_model,
                 num_heads,
                 dff,
                 input_vocab_size,
                 maximum_position_encoding,
                 dropout_rate=0.1):

        super().__init__()

        self.d_model = d_model
        self.num_layers = num_layers
        self.supports_masking = True # Declare support for masking

        # Token Embedding
        self.embedding = tf.keras.layers.Embedding(
            input_vocab_size,
            d_model
        )

        # Positional Encoding
        self.pos_encoding = positional_encoding(
            maximum_position_encoding,
            d_model
        )

        # Stack of Encoder Layers
        self.enc_layers = [
            Encoder_Layer(
                d_model,
                num_heads,
                dff,
                dropout_rate
            )
            for _ in range(num_layers)
        ]

        # Dropout
        self.dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

    def call(self,
             x,
             training=False,
             mask=None):

        seq_len = tf.shape(x)[1]

        # Token Embedding

        x = self.embedding(x)

        # Shape:
        # (batch_size, seq_len, d_model)

        # Scale Embeddings
        x *= tf.math.sqrt(
            tf.cast(self.d_model, tf.float32)
        )

        # Add Positional Encoding

        x += self.pos_encoding[:, :seq_len, :]

        # Dropout

        x = self.dropout(
            x,
            training=training
        )

        # Pass Through N Layers

        for i in range(self.num_layers):

            x, _ = self.enc_layers[i](
                x,
                training=training,
                mask=mask
            )

        # Shape:
        # (batch_size, seq_len, d_model)

        return x

In [91]:
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)  # masked self-attention
        self.mha2 = MultiHeadAttention(d_model, num_heads)  # cross-attention over encoder output
        self.ffn = Feed_Forward_Network(d_model, dff)

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)
        self.dropout3 = tf.keras.layers.Dropout(rate)
        self.supports_masking = True # Declare support for masking

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(attn1 + x)

        attn2, attn_weights_block2 = self.mha2(enc_output, enc_output, out1, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(ffn_output + out2)

        return out3, attn_weights_block1, attn_weights_block2

In [92]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size,
                 max_pos_encoding, rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.supports_masking = True # Declare support for masking

        self.embedding = tf.keras.layers.Embedding(target_vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_pos_encoding, d_model)

        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        seq_len = tf.shape(x)[1]
        attention_weights = {}

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x, block1, block2 = self.dec_layers[i](
                x, enc_output=enc_output, training=training,
                look_ahead_mask=look_ahead_mask, padding_mask=padding_mask
            )
            attention_weights[f"decoder_layer{i + 1}_block1"] = block1
            attention_weights[f"decoder_layer{i + 1}_block2"] = block2

        return x, attention_weights  # (batch, target_seq_len, d_model)

In [93]:
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 input_vocab_size, target_vocab_size,
                 max_pos_encoding_input, max_pos_encoding_target, rate=0.1):
        super().__init__()
        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                                input_vocab_size, max_pos_encoding_input, rate)
        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                                target_vocab_size, max_pos_encoding_target, rate)
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs, training):
        inp, tar = inputs
        enc_padding_mask, look_ahead_mask, dec_padding_mask = create_mask(inp, tar)

        enc_output = self.encoder(inp, training=training, mask=enc_padding_mask)  # (batch, inp_seq_len, d_model)

        dec_output, attention_weights = self.decoder(
            tar, enc_output=enc_output, training=training,
            look_ahead_mask=look_ahead_mask, padding_mask=dec_padding_mask
        )  # (batch, tar_seq_len, d_model)

        final_output = self.final_layer(dec_output)  # (batch, tar_seq_len, target_vocab_size)

        return final_output, attention_weights

In [94]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

    def get_config(self):
        return {"d_model": float(self.d_model.numpy()), "warmup_steps": self.warmup_steps}

In [95]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")


def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))  # ignore padding (token id 0)
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)


def accuracy_function(real, pred):
    accuracies = tf.equal(real, tf.argmax(pred, axis=2, output_type=tf.int32))
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    accuracies = tf.math.logical_and(mask, accuracies)

    accuracies = tf.cast(accuracies, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(accuracies) / tf.reduce_sum(mask)

In [82]:
def create_padding_mask(seq):
  ''' seq shape --> (batch_size , seq_length)
      output of this func dimn -->(batch_size , 1 , 1 , seq_length)
      output -->(batch_size , heads , Query_length , key_length)
       '''

  mask = tf.cast(tf.equal(seq,0),tf.float32)
  return mask[: , tf.newaxis , tf.newaxis , :]


def create_look_ahead_mask(size):
  ''' size --> target sequence length
      output of this func -->(size,size)
      '''
  i = tf.range(size)[:, tf.newaxis]
  j = tf.range(size)

  mask = tf.cast(i < j, tf.float32)

  return mask


def create_mask(inp , tar):
  # encoder padding mask
  enc_padding_mask = create_padding_mask(inp)

  # decoder padding_mask(used in cross attention)
  dec_padding_mask = create_padding_mask(inp)

  # look ahead mask for the 1st attention block (masked MHA)
  look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
  dec_target_padding_mask = create_padding_mask(tar)
  combined_mask = tf.maximum(dec_target_padding_mask , look_ahead_mask)

  return enc_padding_mask , combined_mask , dec_padding_mask

def make_toy_batch(batch_size, seq_len, vocab_size, start_token, end_token):
    """
    Builds a copy task: input is a random sequence of tokens (2..vocab_size-1),
    target is the SAME sequence wrapped with <start> ... <end>.
    Token 0 is reserved for padding.
    """
    inp = np.random.randint(2, vocab_size, size=(batch_size, seq_len))
    tar = np.concatenate(
        [np.full((batch_size, 1), start_token), inp, np.full((batch_size, 1), end_token)],
        axis=1,
    )
    return tf.constant(inp, dtype=tf.int32), tf.constant(tar, dtype=tf.int32)


def main(): # Re-running main to pick up positional_encoding fix and masking support
    # --- toy hyperparameters (small, just to prove the model trains) ---
    num_layers = 2
    d_model = 64
    dff = 128
    num_heads = 4
    dropout_rate = 0.1

    vocab_size = 50       # shared toy vocab for src/tgt
    seq_len = 8
    start_token = 1
    end_token = vocab_size - 1  # ensure inp sampling (2..vocab_size-1) doesn't reuse end_token oddly; fine for toy

    transformer = Transformer(
        num_layers=num_layers,
        d_model=d_model,
        num_heads=num_heads,
        dff=dff,
        input_vocab_size=vocab_size,
        target_vocab_size=vocab_size,
        max_pos_encoding_input=1000,
        max_pos_encoding_target=1000,
        rate=dropout_rate,
    )

    learning_rate = CustomSchedule(d_model, warmup_steps=100)
    optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

    @tf.function
    def train_step(inp, tar):
        tar_inp = tar[:, :-1]   # decoder input (shifted right)
        tar_real = tar[:, 1:]   # what the decoder should predict

        with tf.GradientTape() as tape:
            predictions, _ = transformer([inp, tar_inp], training=True)
            loss = loss_function(tar_real, predictions)

        gradients = tape.gradient(loss, transformer.trainable_variables)
        optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

        acc = accuracy_function(tar_real, predictions)
        return loss, acc

    print("Training toy 'copy sequence' task to verify the implementation...\n")
    for step in range(1, 801):
        inp, tar = make_toy_batch(batch_size=64, seq_len=seq_len, vocab_size=vocab_size,
                                   start_token=start_token, end_token=end_token)
        loss, acc = train_step(inp, tar)
        if step % 100 == 0 or step == 1:
            print(f"step {step:4d}  loss {loss:.4f}  accuracy {acc:.4f}")

    print("\nDone. Accuracy should approach 1.0 as the model learns to copy the input.")


if __name__ == "__main__":
    main()

Training toy 'copy sequence' task to verify the implementation...



/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'multi_head_attention_18' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_6' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'multi_head_attention_19' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the 

step    1  loss 4.2876  accuracy 0.0295
step  100  loss 0.8050  accuracy 0.8090
step  200  loss 0.2476  accuracy 0.9427
step  300  loss 0.2282  accuracy 0.9583
step  400  loss 0.1035  accuracy 0.9774
step  500  loss 0.0750  accuracy 0.9861
step  600  loss 0.1310  accuracy 0.9809
step  700  loss 0.0657  accuracy 0.9896
step  800  loss 0.0420  accuracy 0.9948

Done. Accuracy should approach 1.0 as the model learns to copy the input.
